In [15]:
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import shutil
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [7]:
coco_path = r'../Data/Raw/DENTEX CHALLENGE 2023/Training_Data\quadrant-enumeration-disease\train_quadrant_enumeration_disease.json'
images_path = r'../Data/Raw/DENTEX CHALLENGE 2023/Training_Data/quadrant-enumeration-disease/xrays'

with open(coco_path) as f:
    coco_file = json.load(f)

In [8]:
# Stage 1
s1_train_path = r'..\Data\Processed\Stage 1 (Disease Detection)\train'
s1_val_path = r'..\Data\Processed\Stage 1 (Disease Detection)\val'
s1_test_path = r'..\Data\Processed\Stage 1 (Disease Detection)\test'

# Stage 2
s2_train_path = r'..\Data\Processed\Stage 1 (Disease Classifier)\train'
s2_val_path = r'..\Data\Processed\Stage 1 (Disease Classifier)\val'
s2_test_path = r'..\Data\Processed\Stage 1 (Disease Classifier)\test'

In [ ]:
diagnosis_map = {
    0: "impacted",
    1: "caries",
    2: "periapical",
    3: "deep_caries"
}


def create_dentex_df(diseases_dict):

    f_name = []
    bbox = []
    img_h = []
    img_w = []
    disease = []
    for item in coco_file['images']:
        for ann in coco_file['annotations']:
            if int(ann['image_id']) == int(item['id']):
                f_name.append(item['file_name'])
                bbox.append(ann['bbox'])
                img_h.append(item['height'])
                img_w.append(item['width'])
                disease.append(diseases_dict[int(ann['category_id_3'])])

    return pd.DataFrame({'File_Name':f_name,
                    'Bbox':bbox,
                    'Height':img_h,
                    'Width': img_w,
                    'Disease_Name':disease})

df = create_dentex_df(diagnosis_map)
df

,File_Name,Bbox,Height,Width,Disease_Name
0,train_673.png,"[542.0, 698.0, 220.0, 271.0]",1316,2744,impacted
1,train_673.png,"[1952.0, 693.0, 177.0, 270.0]",1316,2744,impacted
2,train_673.png,"[675.0, 708.0, 243.0, 300.0]",1316,2744,caries
3,train_673.png,"[1463.0, 725.0, 98.0, 425.0]",1316,2744,caries
4,train_673.png,"[1536.0, 753.0, 103.0, 381.0]",1316,2744,caries
...,...,...,...,...,...
3524,train_370.png,"[1851.2857142857142, 474.2857142857142, 117.14...",1316,2948,caries
3525,train_370.png,"[1959.0, 479.9999999999999, 127.0, 274.2857142...",1316,2948,caries
3526,train_370.png,"[2024.9999999999998, 463.0, 147.00000000000023...",1316,2948,deep_caries
3527,train_370.png,"[1921.7142857142856, 749.0, 221.28571428571445...",1316,2948,caries


In [11]:
df.to_csv(r'..\Data\Processed\data (before spliting).csv')

We will do one with class weighting way
and after that we will do other way by doing SMOTE or oversampling

so i will oversampling here and the class weighting way in training process 

In [12]:
df['Disease_Name'].value_counts()

Disease_Name
caries         2189
impacted        604
deep_caries     578
periapical      158
Name: count, dtype: int64

In [35]:
train_df , valid_df = train_test_split(df,test_size=0.1,stratify=df['Disease_Name'],random_state=42)
train_df , test_df = train_test_split(train_df,test_size=0.1,stratify=train_df['Disease_Name'],random_state=42)

In [36]:
train_df

,File_Name,Bbox,Height,Width,Disease_Name
3224,train_422.png,"[597.0, 641.0, 236.0, 161.0]",1316,2836,impacted
2844,train_184.png,"[1973.0, 758.0, 260.0, 295.0]",1504,2872,caries
1701,train_648.png,"[702.0746887966804, 790.8713692946058, 261.410...",1316,2710,caries
805,train_273.png,"[1044.0, 910.0, 155.0, 322.0]",1504,2868,periapical
3122,train_671.png,"[778.0, 740.0, 275.0, 321.0]",1503,2892,caries
...,...,...,...,...,...
1642,train_200.png,"[1124.0, 735.0, 130.0, 242.0]",1316,2936,deep_caries
2930,train_279.png,"[1678.0, 746.0, 148.0, 318.0]",1316,2885,caries
3046,train_633.png,"[2058.0, 772.0, 217.0, 167.0]",1316,2973,impacted
1456,train_11.png,"[920.0, 864.0, 250.0, 332.0]",1504,2872,caries


In [37]:
valid_df

,File_Name,Bbox,Height,Width,Disease_Name
3237,train_258.png,"[874.0, 779.0, 201.0, 264.0]",1504,2872,deep_caries
3453,train_479.png,"[1258.0, 511.0, 123.0, 287.0]",1504,2872,caries
1895,train_557.png,"[1736.0, 637.0, 212.0, 279.0]",1316,2967,caries
1368,train_477.png,"[1734.0, 488.0, 160.0, 426.0]",1504,2888,caries
1858,train_699.png,"[1568.456140350877, 772.8947368421052, 159.649...",1316,2768,caries
...,...,...,...,...,...
618,train_393.png,"[1393.0, 589.0, 50.0, 281.0]",1504,2872,deep_caries
676,train_456.png,"[888.0, 430.0, 176.0, 311.0]",1316,2799,caries
1224,train_681.png,"[1689.0, 663.3333333333334, 161.0, 307.6666666...",1316,2844,caries
2863,train_207.png,"[2020.4545454545455, 286.3636363636364, 123.86...",1316,2796,caries


In [38]:
test_df

,File_Name,Bbox,Height,Width,Disease_Name
3342,train_34.png,"[1871.0, 917.0, 289.0, 324.0]",1504,2872,caries
1856,train_27.png,"[960.0, 359.0, 118.0, 309.0]",1316,2822,caries
2034,train_64.png,"[902.0, 420.0, 158.0, 213.0]",1316,2896,impacted
2708,train_672.png,"[1161.0, 485.0, 102.0, 284.0]",1316,2903,caries
399,train_404.png,"[1870.0, 760.0, 235.0, 238.0]",1316,2960,caries
...,...,...,...,...,...
1845,train_670.png,"[730.6666666666666, 650.6666666666666, 230.666...",1316,2771,caries
617,train_393.png,"[1436.0, 606.0, 66.0, 270.0]",1504,2872,deep_caries
2457,train_239.png,"[2016.705882352941, 515.235294117647, 338.2352...",1316,2955,caries
3033,train_278.png,"[1682.0, 302.0, 110.0, 336.0]",1316,2922,caries


In [39]:
train_df.shape

(2858, 5)

In [40]:
valid_df.shape

(353, 5)

In [41]:
test_df.shape

(318, 5)

In [42]:
train_df['Disease_Name'].value_counts()

Disease_Name
caries         1773
impacted        489
deep_caries     468
periapical      128
Name: count, dtype: int64

In [43]:
valid_df['Disease_Name'].value_counts()

Disease_Name
caries         219
impacted        60
deep_caries     58
periapical      16
Name: count, dtype: int64

In [44]:
test_df['Disease_Name'].value_counts()

Disease_Name
caries         197
impacted        55
deep_caries     52
periapical      14
Name: count, dtype: int64